# 8 · Projected & globe maps

Digital-Earth renders maps in true map **projections** — the orthographic **globe**, Robinson, Mollweide, polar-stereographic — without Cartopy. The recipe: **pyramids** reprojects the data to the display CRS, **digitalearth** assembles the projection boundary + graticule from pyramids' coordinate transforms, and **cleopatra** draws the static frame (boundary, clip, graticule).

Use `Map(crs=projections.<name>(...), globe=True)`, draw layers as usual, then `coastlines()` / `graticule()` / `set_global()`. The frame is applied on `render()` / `save` / `show`.

Natural-Earth context layers work on a globe too: `ocean()` fills the projection disc, `land()` / `lakes()` fill polygons re-closed at the projection limb, and `coastlines()` / `borders()` / `rivers()` add lines — all clipped to the boundary and drawn beneath the data.

**Setup** — imports and a synthetic global field so the projected maps have data covering the domain.

In [ ]:
%matplotlib inline
import numpy as np
from pathlib import Path
from pyramids.dataset import Dataset, GeoReference
from digitalearth.static import Map, projections

# A synthetic GLOBAL field (1-degree lon/lat) so projected maps have data over the whole domain.
ny, nx = 180, 360
lat = np.linspace(90, -90, ny)[:, None]
lon = np.linspace(-180, 180, nx)[None, :]
field = (np.cos(np.deg2rad(lat)) * 30 + 5 * np.sin(np.deg2rad(2 * lon))) * np.ones((ny, nx))
GLOBAL = Dataset.from_array(
    arr=field.astype("float32"),
    geo_ref=GeoReference(geo=(-180.0, 1.0, 0.0, 90.0, 0.0, -1.0), epsg=4326),
)
print("global field:", GLOBAL.rows, "x", GLOBAL.columns)

## Orthographic globe

`projections.orthographic(lon, lat)` centres the globe; `set_global()` shows the whole disc. The far hemisphere reprojects to nodata and is clipped to the boundary.

> **Centre it anywhere:** change `lon`/`lat` to point the globe at any place — see [*Global data on the globe* &rarr; *Centre the globe on a place*](11_global_data_on_globe.ipynb) for a multi-centre grid and a zoom-to-region example.

In [ ]:
m = Map(crs=projections.orthographic(lon=10, lat=25), globe=True, figsize=(7, 7))
m.field(GLOBAL, cmap='YlGnBu_r')
try:
    m.coastlines(resolution='110m')
except Exception as exc:
    print('coastlines offline:', type(exc).__name__)
m.graticule(lon_step=30, lat_step=30)
m.set_global()
m.set_title('Orthographic globe')
m.render()
m.fig;

## Land & ocean fills on a globe

`ocean()` fills the whole projection disc with the ocean colour (exact and network-free), and `land()` / `lakes()` fill Natural-Earth polygons **re-closed at the projection limb** into finite rings. `rivers()` adds line centerlines. Fills are drawn *beneath* the data and coastlines and clipped to the boundary when the frame is applied — so they form a clean base map. (Natural-Earth layers download on first use; this cell falls back to the disc-only ocean fill when offline.)

In [ ]:
m = Map(crs=projections.orthographic(lon=-30, lat=20), globe=True, figsize=(7, 7))
m.ocean()                         # fills the whole projection disc (no network)
try:
    m.land(resolution='110m')     # Natural-Earth land, re-closed at the limb
    m.coastlines(resolution='110m')
except Exception as exc:
    print('natural earth offline:', type(exc).__name__)
m.graticule(lon_step=30, lat_step=30)
m.set_global()
m.set_title('Land & ocean on a globe')
m.render()
m.fig;

## Robinson — whole-world projection

Robinson is a rounded-rectangle world projection (proj4 `+proj=robin`); pyramids 0.27.0 reprojects rasters to it.

In [ ]:
m = Map(crs=projections.robinson(lon=0), globe=True, figsize=(9, 5))
m.field(GLOBAL, cmap='YlGnBu_r')
try:
    m.coastlines(resolution='110m')
except Exception as exc:
    print('coastlines offline:', type(exc).__name__)
m.graticule(lon_step=30, lat_step=30)
m.set_global()
m.set_title('Robinson')
m.render()
m.fig;

## Mollweide — equal-area world projection

In [ ]:
m = Map(crs=projections.mollweide(lon=0), globe=True, figsize=(9, 5))
m.field(GLOBAL, cmap='YlGnBu_r')
try:
    m.coastlines(resolution='110m')
except Exception as exc:
    print('coastlines offline:', type(exc).__name__)
m.graticule(lon_step=30, lat_step=30)
m.set_global()
m.set_title('Mollweide')
m.render()
m.fig;